In [11]:
import requests
import csv
import time

API_KEY = "a92ae18a-de55-499a-89c5-d2e82f12ae8f"
BASE_URL = "https://sandbox-api.piste.gouv.fr/cassation/judilibre/v1.0"
headers = {"KeyId": API_KEY, "accept": "application/json"}

r = requests.get(
            f"{BASE_URL}/search",
            headers=headers,
            params={"jurisdiction" : "cc", 'page':0, 'page_size': 50, 'query' : 'décision administrative'},
        )

data = r.json()
print(data.get('results'))

[{'score': 1, 'highlights': {'text': ['depuis que le juge <em>administratif</em> avait écarté son recours en annulation par une <em>décision</em> irrévocable', "sans mettre en cause son autorité de chose jugée, ni l'autorité attachée aux <em>décisions</em> <em>administratives</em>", "Y] se heurte à l'autorité de chose jugée des <em>décisions</em> rendues par la juridiction <em>administrative</em> et", "non seulement la régularité formelle et procédurale d'une <em>décision</em> <em>administrative</em> mais également son", 'depuis que le juge <em>administratif</em> avait écarté son recours en annulation par une <em>décision</em> irrévocable']}, 'id': '61e7b7e1a41da869de68a294', 'jurisdiction': 'cc', 'chamber': 'civ1', 'number': '17-19.489', 'numbers': ['17-19.489'], 'ecli': 'ECLI:FR:CCASS:2022:C100071', 'formation': 'fs', 'publication': ['b'], 'decision_date': '2022-01-19', 'solution': 'cassation', 'type': 'other', 'summary': "Il résulte de la loi des 16-24 août 1790 et du décret du 16 f

In [73]:
import requests
import pandas as pd
import time
import os

# Configuration de l'API
API_KEY = "a92ae18a-de55-499a-89c5-d2e82f12ae8f"
BASE_URL = "https://sandbox-api.piste.gouv.fr/cassation/judilibre/v1.0"
headers = {"KeyId": API_KEY, "accept": "application/json"}

# SEUIL D'ARRÊT : Nombre de NOUVELLES décisions valides à ajouter lors de CE lancement
MAX_DECISIONS = 700
CSV_FILENAME = "corpus_judiciaire.csv"
PAGE_FILENAME = "last_page.txt"  # Fichier qui stocke le numéro de la page

def fetch_decision_details(doc_id):
    """Récupère le contenu détaillé d'une décision par son ID."""
    try:
        response = requests.get(f"{BASE_URL}/decision", headers=headers, params={'id': doc_id})
        if response.status_code == 200:
            return response.json()
    except Exception as e:
        print(f"Erreur sur {doc_id}: {e}")
    return None

def main():
    # 1. Chargement de l'historique existant
    existing_ids = set()
    existing_data = []
    
    if os.path.exists(CSV_FILENAME):
        try:
            df_existing = pd.read_csv(CSV_FILENAME, encoding='utf-8-sig')
            existing_ids = set(df_existing['id'].astype(str).tolist())
            existing_data = df_existing.to_dict(orient='records')
            print(f"Fichier existant trouvé. {len(existing_ids)} décisions déjà enregistrées.")
        except Exception as e:
            print(f"Erreur lors de la lecture du CSV existant : {e}")
    else:
        print("Aucun fichier existant trouvé. Création d'un nouveau corpus.")

    # 2. Récupération de la dernière page lue
    page = 0
    if os.path.exists(PAGE_FILENAME):
        try:
            with open(PAGE_FILENAME, "r") as f:
                page = int(f.read().strip())
            print(f"Reprise du balayage directement à la page : {page}")
        except:
            print("Impossible de lire le fichier de page, démarrage à la page 0.")

    print(f"Objectif : Extraire {MAX_DECISIONS} nouvelles décisions...")
    
    new_data_list = []
    count_empty_expose = 0
    page_size = 50
    stop_extraction = False
    
    while True:
        print(f"Recherche - Page {page}...")
        search_params = {
            "jurisdiction": "cc", 
            "page_size": page_size, 
            "page": page,
            "query": "redressement"
        }
        
        r = requests.get(f"{BASE_URL}/search", headers=headers, params=search_params)
        
        if r.status_code != 200:
            print(f"Erreur lors de la recherche à la page {page}: {r.status_code}")
            break
            
        results = r.json().get('results', [])
        
        if not results:
            print("Fin du balayage : plus aucun résultat disponible sur l'API.")
            break
            
        for item in results:
            if len(new_data_list) >= MAX_DECISIONS:
                print(f"Quota de {MAX_DECISIONS} nouvelles décisions atteint. Arrêt.")
                stop_extraction = True
                break
                
            doc_id = str(item.get('id'))
            
            if doc_id in existing_ids:
                continue
                
            print(f"Nouveau document trouvé ! Traitement de {doc_id} ({len(new_data_list) + 1}/{MAX_DECISIONS})...")
            
            detail = fetch_decision_details(doc_id)
            if detail:
                full_text = detail.get('text', '')
                zones = detail.get('zones', {})
                
                expose = ""
                if 'expose' in zones:
                    m = zones['expose'][0]
                    expose = full_text[m['start']:m['end']].strip()
                
                if not expose:
                    count_empty_expose += 1
                    continue 
                
                new_data_list.append({
                    "id": str(detail.get('id')),
                    "solution": detail.get('solution'),
                    'expose': expose
                })
                
                time.sleep(0.3)
                
        if stop_extraction:
            break
            
        page += 1
    
    try:
        with open(PAGE_FILENAME, "w") as f:
            f.write(str(page))
        print(f"Index de progression sauvegardé : Page {page}")
    except Exception as e:
        print(f"Impossible de sauvegarder la page courante : {e}")
        
    print("-" * 40)
    print(f"Nouvelles décisions ajoutées lors de cette session : {len(new_data_list)}")
    
    # 4. Fusion et Sauvegarde du CSV
    if new_data_list:
        final_data = existing_data + new_data_list
        df_final = pd.DataFrame(final_data)
        df_final.to_csv(CSV_FILENAME, index=False, encoding='utf-8-sig')
        print(f"Fichier '{CSV_FILENAME}' mis à jour. Total global : {len(df_final)} décisions.")
    else:
        print("Aucune nouvelle décision ajoutée. Le fichier reste inchangé.")

if __name__ == "__main__":
    main()

Fichier existant trouvé. 9218 décisions déjà enregistrées.
Reprise du balayage directement à la page : 149
Objectif : Extraire 700 nouvelles décisions...
Recherche - Page 149...
Nouveau document trouvé ! Traitement de 5fca5d90baa43d3ff8ebb746 (1/700)...
Nouveau document trouvé ! Traitement de 613728e3cd58014677433499 (1/700)...
Nouveau document trouvé ! Traitement de 613724bdcd58014677417f99 (1/700)...
Nouveau document trouvé ! Traitement de 627ca5c4ce7765057d218550 (1/700)...
Nouveau document trouvé ! Traitement de 6137279fcd5801467742ce21 (1/700)...
Nouveau document trouvé ! Traitement de 61372735cd5801467742ac3d (1/700)...
Nouveau document trouvé ! Traitement de 61372371cd58014677409da2 (1/700)...
Nouveau document trouvé ! Traitement de 613726f9cd5801467742984e (1/700)...
Nouveau document trouvé ! Traitement de 613722d9cd58014677402413 (1/700)...
Nouveau document trouvé ! Traitement de 613722c2cd58014677401237 (1/700)...
Nouveau document trouvé ! Traitement de 613722c7cd580146774015

In [74]:
import pandas as pd
from sklearn.model_selection import train_test_split

CSV_FILENAME = "corpus_judiciaire.csv"

def clean_and_encode_solutions(df):
    """
    Nettoie la colonne 'solution' et encode :
    Rejet -> 0
    Cassation -> 1
    """
    # 1. Copie pour éviter les warnings Pandas
    df_clean = df.copy()
    
    # 2. Nettoyage du texte (minuscules, suppression des espaces inutiles)
    df_clean['solution'] = df_clean['solution'].astype(str).str.lower().str.strip()
    
    # 3. Filtrage : On ne garde que les lignes qui contiennent 'rejet' ou 'cassation'
    # (Certains arrêts contiennent "cassation partielle" ou "cassation totale", on les regroupe)
    mask_rejet = df_clean['solution'].str.contains('rejet', na=False)
    mask_cassation = df_clean['solution'].str.contains('cassation', na=False)
    
    # On applique le filtre
    df_filtered = df_clean[mask_rejet | mask_cassation].copy()
    
    # 4. Encodage numérique (0 pour Rejet, 1 pour Cassation)
    # On utilise np.where : si 'rejet' est dans le texte -> 0, sinon -> 1 (donc cassation)
    import numpy as np
    df_filtered['label'] = np.where(df_filtered['solution'].str.contains('rejet'), 0, 1)
    
    print(f"Données d'origine : {len(df)} lignes.")
    print(f"Données après filtrage (uniquement Rejet/Cassation) : {len(df_filtered)} lignes.")
    print(f"Répartition - Rejets (0) : {sum(df_filtered['label'] == 0)} | Cassations (1) : {sum(df_filtered['label'] == 1)}")
    
    return df_filtered

def main():
    # Charger le fichier CSV
    try:
        df = pd.read_csv(CSV_FILENAME, encoding='utf-8-sig')
    except FileNotFoundError:
        print(f"Erreur : Le fichier {CSV_FILENAME} n'existe pas encore. Lancez d'abord votre script de collecte.")
        return

    # Vérifier que les colonnes nécessaires sont présentes
    if 'solution' not in df.columns or 'expose' not in df.columns:
        print("Erreur : Le CSV doit contenir les colonnes 'solution' et 'expose'.")
        return

    # Encoder les données
    df_encoded = clean_and_encode_solutions(df)
    
    # Sauvegarder le dataset propre et encodé
    encoded_filename = "corpus_pret_pour_bert.csv"
    df_encoded.to_csv(encoded_filename, index=False, encoding='utf-8-sig')
    print(f"Dataset encodé sauvegardé sous : '{encoded_filename}'")
    
    # --- BONUS : Préparation des splits pour CamemBERT ---
    # Séparation en Train (80%) et Test (20%) pour l'évaluation future du modèle
    train_df, test_df = train_test_split(
        df_encoded, 
        test_size=0.2, 
        random_state=42, 
        stratify=df_encoded['label'] # Maintient la même proportion de 0 et 1 dans les deux sets
    )
    
    print(f"Split terminé : {len(train_df)} échantillons d'entraînement, {len(test_df)} échantillons de test.")

if __name__ == "__main__":
    main()

Données d'origine : 9918 lignes.
Données après filtrage (uniquement Rejet/Cassation) : 9666 lignes.
Répartition - Rejets (0) : 4792 | Cassations (1) : 4874
Dataset encodé sauvegardé sous : 'corpus_pret_pour_bert.csv'
Split terminé : 7732 échantillons d'entraînement, 1934 échantillons de test.


In [62]:
%pip install transformers datasets scikit-learn accelerate evaluate

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 24.5 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 18.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 19.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 20.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 15.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.8/150.8 MB 17.2 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 28.9 MB/s  0:00:00
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25/25 [evaluate]/25 [transformers]ub]
Note: you may need to restart the kernel to use updated packages.


In [63]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import CamembertTokenizer, CamembertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import evaluate
import numpy as np

# 1. Configuration du matériel (Utilise la carte graphique GPU si disponible, sinon le processeur)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Entraînement sur : {device}")

# 2. Chargement du dataset encodé (généré à l'étape précédente)
try:
    df = pd.read_csv("corpus_pret_pour_bert.csv")
except FileNotFoundError:
    print("Erreur : Fichier 'corpus_pret_pour_bert.csv' introuvable.")
    exit()

# Hugging Face attend des colonnes spécifiques : 'text' (les données) et 'label' (les cibles)
df = df.rename(columns={'expose': 'text'})
df = df[['text', 'label']].dropna() # Sécurité au cas où il y aurait des lignes vides

# Split Train (80%) / Validation (20%)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

# Conversion en objet Dataset (le format natif de Hugging Face)
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

# 3. Tokenisation avec CamemBERT
# Le tokenizer découpe le texte juridique en "sub-words" que le modèle comprend
model_name = "camembert-base" # Vous pouvez remplacer par un modèle Legal-BERT si vous préférez
tokenizer = CamembertTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # max_length=512 car c'est la limite physique de CamemBERT. Les longs exposés seront tronqués.
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

# 4. Chargement du modèle CamemBERT pour de la classification binaire (num_labels=2)
model = CamembertForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to(device)

# 5. Métriques d'évaluation (Précision et F1-Score)
metric = evaluate.load("f1")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 6. Configuration des arguments d'entraînement
training_args = TrainingArguments(
    output_dir="./results_camembert",       # Dossier de sauvegarde du modèle
    learning_rate=2e-5,                     # Un taux d'apprentissage très bas pour du fine-tuning
    per_device_train_batch_size=8,          # Ajustez à 4 ou 16 selon la mémoire (VRAM) de votre machine
    per_device_eval_batch_size=8,
    num_train_epochs=3,                     # 3 époques suffisent généralement pour commencer
    weight_decay=0.01,
    eval_strategy="epoch",            # Évaluation à la fin de chaque époque
    save_strategy="epoch",                  # Sauvegarde à chaque époque
    load_best_model_at_end=True,            # Garde le meilleur modèle à la fin
    logging_dir="./logs",
    logging_steps=50,
)

# 7. Création du Trainer et lancement
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

print("Début du fine-tuning...")
trainer.train()

# 8. Sauvegarde finale du modèle entraîné
model.save_pretrained("./mon_camembert_juridique")
tokenizer.save_pretrained("./mon_camembert_juridique")
print("Modèle fine-tuné sauvegardé avec succès dans './mon_camembert_juridique' !")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/arthur/Library/Python/3.11/lib/python/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/arthur/Library/Python/3.11/lib/python/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/arthur/Library/Python/3.11/lib/python/site-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_loop.start(

Entraînement sur : cpu


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/811k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.40M [00:00<?, ?B/s]

Map:   0%|          | 0/3911 [00:00<?, ? examples/s]

Map:   0%|          | 0/978 [00:00<?, ? examples/s]

ImportError: 
CamembertForSequenceClassification requires the PyTorch library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.
